# Species tree distance matrix by branch count

This notebook computes a pairwise distance matrix from a Newick tree where the
distance between two taxa is the **number of branch points (internal nodes)**
on the path connecting them — not the sum of branch lengths.

For example, given the tree `(A, (B, C))`:
- B ↔ C = 1 (one shared ancestor between them)
- A ↔ B = 2 (root + the B/C ancestor)
- A ↔ C = 2

The algorithm is adapted from Wandrille's efficient postorder traversal method
used in notebook 105, but counts edges rather than summing branch lengths,
then subtracts 1 to convert edge count to branch-point count.

In [ ]:
import ete3
import pandas as pd
import numpy as np

In [ ]:
# Load the tree.
tree = ete3.Tree("output/cleaned_trees/Actinopterygii_species_with_order.nwk", format=1, quoted_node_names=True)

In [ ]:
# Sanity check: no duplicated tip names.
tip_names = [leaf.name for leaf in tree.iter_leaves()]
duplicated_names = set([name for name in tip_names if tip_names.count(name) > 1])
print("Duplicated names:", duplicated_names)

In [ ]:
# Compute the branch-count distance matrix using a postorder traversal.
#
# For each internal node we track, for every leaf in its subtree, how many
# edges separate that leaf from the current node.  When two leaves meet at
# their LCA the total edge count on the path is the sum of their individual
# edge counts.  Subtracting 1 converts edge count to branch-point count
# (number of internal nodes traversed on the path).

leaf_to_pos = {l: i for i, l in enumerate(tree.get_leaf_names())}
n_leaves = len(leaf_to_pos)
DMat = np.zeros((n_leaves, n_leaves))

for n in tree.traverse('postorder'):

    if n.is_leaf():
        n.dist_dict = {n.name: 0}
    else:
        n.dist_dict = {}

        # Propagate edge counts upward: each leaf gains +1 edge for the
        # connection between child c and the current node n.
        for c in n.children:
            for l, d in c.dist_dict.items():
                n.dist_dict[l] = d + 1

        # For leaves in different child subtrees, n is their LCA.
        # edge_count = n.dist_dict[l1] + n.dist_dict[l2]
        # branch_point_count = edge_count - 1
        for i, c1 in enumerate(n.children):
            for j, c2 in enumerate(n.children):
                if j <= i:
                    continue
                for l1 in c1.dist_dict.keys():
                    for l2 in c2.dist_dict.keys():
                        d = n.dist_dict[l1] + n.dist_dict[l2] - 1
                        DMat[leaf_to_pos[l1], leaf_to_pos[l2]] = d
                        DMat[leaf_to_pos[l2], leaf_to_pos[l1]] = d

        # Free memory: child dicts are no longer needed.
        for c in n.children:
            c.dist_dict = None

branch_count_dist_matrix = pd.DataFrame(DMat, index=tree.get_leaf_names(), columns=tree.get_leaf_names())
print(f"Distance matrix shape: {branch_count_dist_matrix.shape}")

In [ ]:
# Save.
branch_count_dist_matrix.to_csv("output/Actinopterygii_species_branch_count_distance_matrix.csv")
print("Saved to output/Actinopterygii_species_branch_count_distance_matrix.csv")

# Deallocate DMat
DMat = None